In [ ]:
import wrds
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from datetime import datetime


db = wrds.Connection() 

In [ ]:
def plot_financial_engine(df, idx):
    fig, axes = plt.subplots(4, 1, figsize=(12, 22))
    
    
    plot_configs = [
        (['roa', 'roe', 'roc'], 'Profitability Metrics (ROA/ROE/ROC)', 0),
        (['current_ratio', 'quick_ratio'], 'Liquidity Analysis (Current/Quick)', 1),
        (['roa_ma'], 'ROA 4-Quarter Moving Average (Trend)', 2),
        (['ret'], 'Quarterly Returns (%)', 3)
    ]
    
    for cols, title, i in plot_configs:
        for col in cols:
            axes[i].plot(df['datadate'], df[col], label=col.upper(), marker='o', markersize=4, alpha=0.7)
            
            axes[i].scatter(df.loc[idx, 'datadate'], df.loc[idx, col], 
                            color='red', s=130, edgecolors='black', zorder=10)
        
        
        if i == 0:
            axes[i].axhline(y=0.05, color='gray', linestyle='--', label='Industry Baseline (5%)')
            
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].legend()
        axes[i].grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()


def calculate_risk_metrics(df, idx):
    recent_rets = df.loc[idx:, 'ret'].dropna()
    if len(recent_rets) < 2:
        print("Note: Insufficient data points after the selected date to calculate risk metrics.")
       
        return
        
    
    vol = recent_rets.std() * np.sqrt(4)
    sharpe = (recent_rets.mean() / recent_rets.std()) * np.sqrt(4) if recent_rets.std() != 0 else 0
    
    
   

  
    print(f"\n--- 📊 Risk & Performance Metrics (From Selected Date) ---")
    print(f"📈 Annualized Volatility: {vol:.2%}")
    print(f"⚖️ Risk-Adjusted Return (Sharpe Ratio): {sharpe:.2f}")
    print("-" * 50)


In [ ]:
ticker_input = widgets.Text(value='NVDA', description='US Ticker:', placeholder='For Example: TSLA')
date_input = widgets.DatePicker(
    description='Select Date:',
    value=datetime(2024, 4, 8).date(),
    min=datetime(2021, 1, 1).date(),
    max=datetime(2026, 1, 1).date()
)
btn = widgets.Button(description="Generate Deep Value Report", button_style='danger')
out = widgets.Output()


        

def on_click_analyze(b):
    with out:
        clear_output(wait=True)
        ticker = ticker_input.value.strip().upper()
        sel_date = pd.to_datetime(date_input.value).normalize()

        query = f"""
        SELECT datadate, 
               prccq, 
               (niq / NULLIF(atq, 0)) as roa,
               (niq / NULLIF(ceqq, 0)) as roe,
               (oiadpq / NULLIF(atq - lctq, 0)) as roc,
               (actq / NULLIF(lctq, 0)) as current_ratio,
               ((cheq + rectq) / NULLIF(lctq, 0)) as quick_ratio
        FROM comp.fundq  
        WHERE tic = '{ticker}' 
          AND datadate >= '2020-01-01' 
          AND datadate <= '2026-01-01'
          AND indfmt = 'INDL' 
          AND datafmt = 'STD' 
          AND popsrc = 'D'    
          AND consol = 'C'    
        ORDER BY datadate
        """



        try:
            df = db.raw_sql(query)
            if df is None or df.empty:
                print(f"❌ No financial data found for {ticker}.")
                return

            df['datadate'] = pd.to_datetime(df['datadate'])
            df['roa_ma'] = df['roa'].rolling(window=4).mean()
            df['ret'] = df['prccq'].pct_change()

            idx = (df['datadate'] - sel_date).abs().idxmin()
            latest = df.loc[idx]

            print(f"🚀 Alpha Insight Engine | Report Target: {ticker}")
            print(f"📅 Selected Quarter: {latest['datadate'].date()}")
            print("-" * 40)

            if latest['current_ratio'] < 1.2:
                print("🔴 Solvency Alert: Current Ratio is below 1.2. Please monitor short-term cash flow.")
            else:
                print("🟢 Financial Health: Strong liquidity with solid short-term solvency.")

            post_data = df.loc[idx:]
            if len(post_data) > 1:
                total_gain = (post_data['prccq'].iloc[-1] / latest['prccq'] - 1) * 100
                print(f"💰 Value Simulation: Total return since entry at this point is {total_gain:.2f}%")

            plot_financial_engine(df, idx)
            calculate_risk_metrics(df, idx)

        except Exception as e:
            print(f"⚠️ Error occurred during analysis: {e}")

btn.on_click(on_click_analyze)
display(widgets.VBox([ticker_input, date_input, btn]), out)


In [ ]:
db.close()